# Métodos de agrupamiento

## Agrupamiento por mezclas gaussianas

In [1]:
import pandas as pd
from google.colab import files

# Métodos de agrupamiento
# Algoritmos de agrupamiento por k-medias.

# Cargue del conjunto de datos
uploaded = files.upload()
datos = pd.read_excel("HOMA IR in overweight obesity.xlsx")

print(datos.head())
print(datos.shape)

Saving HOMA IR in overweight obesity.xlsx to HOMA IR in overweight obesity.xlsx
   Unnamed: 0  No  Age  Age Category  Gender  GenderCategory  Body weight  \
0         NaN   1   32             1  female               2         72.5   
1         NaN   2   35             1    Male               1        135.0   
2         NaN   3   30             1    Male               1        123.8   
3         NaN   4   34             1  female               2         97.4   
4         NaN   5   28             1  female               2         76.6   

   body height        BMI Obesity category  ...  Gait speed  \
0        157.0  29.412958  Grade 1 Obesity  ...         1.1   
1        180.0  41.666667  Grade 2 Obesity  ...         1.0   
2        173.0  41.364563  Grade 2 Obesity  ...         1.0   
3        165.0  35.775941  Grade 2 Obesity  ...         1.1   
4        150.0  34.044444  Grade 2 Obesity  ...         1.2   

  gait speed category gait speed cat no  BIA BIA Category  BIA Cat no  \
0    

In [2]:
datos = datos[
    [
        "Obesity category", # Sarcopenia: Normal, sarcpenia (Presarcopenia), Sarcopenia, severe sarcopenia (Severa),
        "Age", # Edad (en años cumplidos)
        "BMI", # Índice de masa corporal (en kg/m², Peso/Estatura(mts)^2)
        "Hand grip test", # Prueba de fuerza de agarre manual (en kg)
        "Gait speed", # Velocidad de la marcha (en m/s)
        "BIA", # Análisis de Impedancia Bioeléctrica (en kg/m²)
        "HOMA IR" # Resistencia a la Insulina (adimensional)
    ]
].copy()

datos["Obesity category"] = pd.Categorical(
    datos["Obesity category"],
    categories=["Overweight", "Grade 1 Obesity", "Grade 2 Obesity"]
).rename_categories(["Overweight", "Grade 1", "Grade 2"])

datos = datos.rename(
    columns={
        "Hand grip test": "HGT",
        "Gait speed": "Gait",
        "HOMA IR": "IR",
        "Obesity category": "Obesity"
    }
)

datos = datos.dropna()

print(datos.head())
print(datos.shape)

   Obesity  Age        BMI   HGT  Gait  BIA   IR
0  Grade 1   32  29.412958  27.5   1.1  5.6  1.3
1  Grade 2   35  41.666667  39.2   1.0  7.2  5.3
2  Grade 2   30  41.364563  43.0   1.0  7.3  3.6
3  Grade 2   34  35.775941  27.4   1.1  4.0  1.2
4  Grade 2   28  34.044444  23.9   1.2  5.5  2.1
(100, 7)


In [3]:
# Variable de grupo a clasificar
datos["Obesity"] = datos["Obesity"].astype("category")

print(datos["Obesity"].value_counts().sort_index())

y = "Obesity"

# Predictores
xvars = datos.columns.difference([y]).tolist()

xvars

Obesity
Overweight    26
Grade 1       43
Grade 2       31
Name: count, dtype: int64


['Age', 'BIA', 'BMI', 'Gait', 'HGT', 'IR']

In [4]:
from sklearn.model_selection import train_test_split

# Partición estratificada entrenamiento/prueba
part = 0.70 # proporción muestra de entrenamiento

idx_train, idx_test = train_test_split(
    range(len(datos)),
    train_size=part,
    stratify=datos[y],
    random_state=123
)

idx_train

train = datos.iloc[idx_train].copy() # muestra de entrenamiento
test  = datos.iloc[idx_test].copy()  # muestra de prueba

print(train.shape)
print(test.shape)

(70, 7)
(30, 7)


In [5]:
# Tamaños
print(datos[y].value_counts().sort_index())
print(train[y].value_counts().sort_index())
print(test[y].value_counts().sort_index())

Obesity
Overweight    26
Grade 1       43
Grade 2       31
Name: count, dtype: int64
Obesity
Overweight    18
Grade 1       30
Grade 2       22
Name: count, dtype: int64
Obesity
Overweight     8
Grade 1       13
Grade 2        9
Name: count, dtype: int64


In [6]:
import numpy as np

# Revisión inicial
ng = train[y].value_counts().sort_index()
print(ng)

p = len(xvars)
print(p)

ng > p # preferible

Obesity
Overweight    18
Grade 1       30
Grade 2       22
Name: count, dtype: int64
6


,count
Obesity,
Overweight,True
Grade 1,True
Grade 2,True


In [7]:
# Colinealidad (¿k < 30?)
R = train[xvars].corr()
print(R)

kappa_R = np.linalg.cond(R)
kappa_R

           Age       BIA       BMI      Gait       HGT        IR
Age   1.000000 -0.106514 -0.606020 -0.363189 -0.356513 -0.113102
BIA  -0.106514  1.000000 -0.077900  0.120097  0.455768 -0.259515
BMI  -0.606020 -0.077900  1.000000  0.248447  0.320556  0.274947
Gait -0.363189  0.120097  0.248447  1.000000  0.195897 -0.028565
HGT  -0.356513  0.455768  0.320556  0.195897  1.000000 -0.198413
IR   -0.113102 -0.259515  0.274947 -0.028565 -0.198413  1.000000


np.float64(6.612995182441637)

In [8]:
# Estandarización usando solo entrenamiento
medias = train[xvars].mean()
desv = train[xvars].std(ddof=1)

train_st = train.copy()
test_st = test.copy()

train_st[xvars] = (train[xvars] - medias) / desv
test_st[xvars] = (test[xvars] - medias) / desv

print(train_st.head())
print(test_st.head())

       Obesity       Age       BMI       HGT      Gait       BIA        IR
76     Grade 1 -1.275827 -0.439378  0.420521 -0.914644 -0.039166 -0.762567
39     Grade 1  0.463937 -0.234843 -0.836959 -0.914644 -0.577393 -0.194497
52  Overweight  0.391447 -0.874836  0.020414 -0.914644 -0.280601 -0.194497
89     Grade 1  0.028996 -0.275344 -0.379694  0.173293  0.028112 -0.194497
81     Grade 2  0.173976  0.876663  0.763470  1.642007 -0.241002  1.377165
    Obesity       Age       BMI       HGT      Gait       BIA        IR
86  Grade 1  0.826388 -0.753392 -0.151061  1.261229  0.566339 -0.440660
43  Grade 1 -1.275827 -0.079444 -0.253946  0.173293  0.519244 -0.554275
6   Grade 2 -1.420807  1.182906  2.592532  0.173293  1.239122 -0.099818
0   Grade 1 -1.783257  0.156395  0.477679  0.717261 -0.173723 -1.141281
3   Grade 2 -1.638277  1.321523  0.466248  0.717261 -1.250177 -1.235959


In [9]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis

# Ajuste del análisis discriminante

# Fórmula
formula_ad = f"{y} ~ ."
formula_ad

# Método
metodo = "qda" # "lda", "qda"

X_train = train_st[xvars]
y_train = train_st[y]

if metodo == "lda":
    mod_ad = LinearDiscriminantAnalysis().fit(X_train, y_train)

elif metodo == "qda":
    mod_ad = QuadraticDiscriminantAnalysis().fit(X_train, y_train)

else:
    raise ValueError("En sklearn estándar solo se traducen directamente lda y qda.")

In [10]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict

# Validación cruzada K-fold estratificada
K = min(10, train_st[y].value_counts().min()) ; K # folds

cv = StratifiedKFold(
    n_splits=K,
    shuffle=True,
    random_state=123
)

if metodo == "lda":
    cv_ad = LinearDiscriminantAnalysis()

elif metodo == "qda":
    cv_ad = QuadraticDiscriminantAnalysis()

else:
    raise ValueError("Este método no se traduce directamente con sklearn estándar.")

pred_cv = cross_val_predict(
    cv_ad,
    train_st[xvars],
    train_st[y],
    cv=cv
)

# Matriz de confusión
MC_cv_ad = pd.crosstab(
    pd.Series(train_st[y].to_numpy(), name="Real"),
    pd.Series(pred_cv, name="Predicho")
)

MC_cv_ad

Predicho,Grade 1,Grade 2,Overweight
Real,,,
Grade 1,23,7,0
Grade 2,6,16,0
Overweight,3,2,13


In [11]:
# Exactitud, mal clasificación, sensibilidad y exactitud balanceada
accuracy_cv_ad = np.trace(MC_cv_ad.to_numpy()) / MC_cv_ad.to_numpy().sum()
accuracy_cv_ad

tmc_cv_ad = 1 - accuracy_cv_ad
tmc_cv_ad

sens_cv_ad = pd.Series(
    np.diag(MC_cv_ad.to_numpy()) / MC_cv_ad.sum(axis=1).to_numpy(),
    index=MC_cv_ad.index
)
sens_cv_ad

balanced_cv_ad = sens_cv_ad.mean()
balanced_cv_ad

np.float64(0.7387205387205388)

In [12]:
# Sensibilidad, Precisión y F1
sens_cv_ad = pd.Series(
    np.diag(MC_cv_ad.to_numpy()) / MC_cv_ad.sum(axis=1).to_numpy(),
    index=MC_cv_ad.index
)
sens_cv_ad # qué tanto detecta bien un grupo real

prec_cv_ad = pd.Series(
    np.diag(MC_cv_ad.to_numpy()) / MC_cv_ad.sum(axis=0).to_numpy(),
    index=MC_cv_ad.columns
)
prec_cv_ad # qué tan confiable es una asignación a cada grupo

f1_cv_ad = 2 * sens_cv_ad * prec_cv_ad / (sens_cv_ad + prec_cv_ad)
f1_cv_ad # equilibrio entre sensibilidad y precisión

f1_macro_cv_ad = f1_cv_ad.mean(skipna=True)
f1_macro_cv_ad # equilibrio entre sensibilidad y precisión general

np.float64(0.7538320750400365)

In [13]:
# Predicción en prueba
pred_ad = mod_ad.predict(test_st[xvars])

# Evaluación final en prueba
MC_test_ad = pd.crosstab(
    pd.Series(test_st[y].to_numpy(), name="Real"),
    pd.Series(pred_ad, name="Predicho")
)

MC_test_ad

Predicho,Grade 1,Grade 2,Overweight
Real,,,
Grade 1,11,2,0
Grade 2,0,9,0
Overweight,2,1,5


In [14]:
# Exactitud, mal clasificación, sensibilidad y exactitud balanceada
niveles = test_st[y].cat.categories

MC_test_ad = MC_test_ad.reindex(
    index=niveles,
    columns=niveles,
    fill_value=0
)

accuracy_test_ad = np.trace(MC_test_ad.to_numpy()) / MC_test_ad.to_numpy().sum()
accuracy_test_ad

tmc_test_ad = 1 - accuracy_test_ad
tmc_test_ad

sens_test_ad = pd.Series(
    np.diag(MC_test_ad.to_numpy()) / MC_test_ad.sum(axis=1).to_numpy(),
    index=MC_test_ad.index
)
sens_test_ad

balanced_test_ad = sens_test_ad.mean()
balanced_test_ad

np.float64(0.8237179487179488)

In [15]:
# Sensibilidad, Precisión y F1
sens_test_ad = pd.Series(
    np.diag(MC_test_ad.to_numpy()) / MC_test_ad.sum(axis=1).to_numpy(),
    index=MC_test_ad.index
)
sens_test_ad # qué tanto detecta bien un grupo real

prec_test_ad = pd.Series(
    np.diag(MC_test_ad.to_numpy()) / MC_test_ad.sum(axis=0).to_numpy(),
    index=MC_test_ad.columns
)
prec_test_ad # qué tan confiable es una asignación a cada grupo

f1_test_ad = 2 * sens_test_ad * prec_test_ad / (sens_test_ad + prec_test_ad)
f1_test_ad # equilibrio entre sensibilidad y precisión

f1_macro_test_ad = f1_test_ad.mean(skipna=True)
f1_macro_test_ad # equilibrio entre sensibilidad y precisión general

np.float64(0.8241758241758242)

In [16]:
# Resumen para comparar con otro método
resumen_ad = pd.DataFrame({
    "Metodo": [metodo],
    "Accuracy_CV": [accuracy_cv_ad],
    "TMC_CV": [tmc_cv_ad],
    "Balanced_CV": [balanced_cv_ad],
    "Accuracy_Test": [accuracy_test_ad],
    "TMC_Test": [tmc_test_ad],
    "Balanced_Test": [balanced_test_ad]
})

resumen_ad

,Metodo,Accuracy_CV,TMC_CV,Balanced_CV,Accuracy_Test,TMC_Test,Balanced_Test
0,qda,0.742857,0.257143,0.738721,0.833333,0.166667,0.823718


In [17]:
# Datos completos etiquetados

# Estandarización usando toda la muestra etiquetada
medias_final = datos[xvars].mean()
desv_final = datos[xvars].std(ddof=1)

datos_final = datos.copy()

datos_final[xvars] = (datos[xvars] - medias_final) / desv_final

datos_final.head()

,Obesity,Age,BMI,HGT,Gait,BIA,IR
0,Grade 1,-1.618657,0.154090,0.376340,0.67240,-0.243942,-0.680946
1,Grade 2,-1.415223,2.493216,1.678707,0.16029,0.882759,1.227401
2,Grade 2,-1.754280,2.435547,2.101698,0.16029,0.953178,0.416354
3,Grade 2,-1.483035,1.368728,0.365208,0.67240,-1.370643,-0.728654
4,Grade 2,-1.889903,1.038200,-0.024389,1.18451,-0.314361,-0.299276


In [18]:
# Regla discriminante final
# función lda, qda, mda, fda y rda
X_final = datos_final[xvars]
y_final = datos_final[y]

if metodo == "lda":
    mod_ad_final = LinearDiscriminantAnalysis().fit(X_final, y_final)

elif metodo == "qda":
    mod_ad_final = QuadraticDiscriminantAnalysis().fit(X_final, y_final)

else:
    raise ValueError("Este método no se traduce directamente con sklearn estándar.")

# Nuevos individuos
nuevos = pd.DataFrame({
    "Age":  [30, 50, 60],
    "BMI":  [41.5, 31.2, 30.5],
    "HGT":  [45.1, 17.4, 22.8],
    "Gait": [1.0, 1.0, 0.8],
    "BIA":  [6.7, 5.3, 7.9],
    "IR":   [3.0, 1.7, 2.1]
})

# Estandarización
nuevos_st = nuevos.copy()
nuevos_st[xvars] = (nuevos[xvars] - medias_final) / desv_final

# Asignación de los nuevos individuos
pred_nuevos = mod_ad_final.predict(nuevos_st[xvars])

pred_nuevos

array(['Grade 2', 'Grade 2', 'Grade 2'], dtype=object)